# MIMIC-IV bridge analysis for the Texas replication

Run all cells (Runtime → Run all). About 2–3 minutes. It reuses the paper's settings, data load and variable preparation, then re-estimates the Medicaid–private gap with Texas-comparable definitions (whole-stay calendar-day LOS, linear probability models, all ICU survivors and the invasive-ventilation cohort). Downloads `mimic_bridge_results.xlsx` (aggregate only).

In [ ]:
# ---- Settings ---------------------------------------------------------------
PROJECT  = "ctheg-scores"
SEED     = 20261112
B        = 500     # FINAL run: bootstrap reps for primary estimates
B_SENS   = 200     # FINAL run: reps for sensitivity / 65+ / Oaxaca / by-destination
OUTFILE  = "aim1_results.xlsx"
MIN_CELL = 11      # aggregate counts below this are suppressed in exported tables

import time, warnings
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # convergence is checked explicitly below

class NotConverged(Exception):
    pass

def check(res, label):
    ok = res.mle_retvals.get("converged", True) if hasattr(res, "mle_retvals") and res.mle_retvals else getattr(res, "converged", True)
    if not ok:
        raise NotConverged(label)
    return res
pd.set_option("display.width", 200, "display.max_columns", 50)
print("statsmodels", statsmodels.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ---- Load the analysis views from BigQuery (no file downloads) ---------------
from google.colab import auth
auth.authenticate_user()          # sign in with the Google account linked to PhysioNet
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

COLS = """ins_cat, dc_group, post_icu_days, any_post_icu_stay, dc_ama, dc_acute_transfer,
          time_valid_strict, age_at_admit, gender, race_cat, sofa_day1, cci_no_age,
          adm_type_cat, invasive_vent, vasopressor, icu_los_days_total, first_careunit,
          anchor_year_group, language_cat"""

raw18 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_primary_18_64`").to_dataframe()
raw65 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_older_65plus`").to_dataframe()
print("18-64:", raw18.shape, "| 65+:", raw65.shape)

In [ ]:
# ---- Variable preparation -----------------------------------------------------
DC_LEVELS = ["home", "home_health", "snf", "rehab", "ltach", "other"]   # code 0 = home (reference)

ICU_MAP = {
    "Cardiac Vascular Intensive Care Unit (CVICU)":       "Cardiac",
    "Coronary Care Unit (CCU)":                           "Cardiac",
    "Medical Intensive Care Unit (MICU)":                 "Medical",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)":   "Mixed",
    "Surgical Intensive Care Unit (SICU)":                "Surgical/Trauma",
    "Trauma SICU (TSICU)":                                "Surgical/Trauma",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)":    "Neuro",
    "Neuro Intermediate":                                 "Neuro",
    "Neuro Stepdown":                                     "Neuro",
}   # anything else (PACU, Medicine, 'Intensive Care Unit (ICU)', small surgical units; ~60 patients) -> 'Mixed'
#     (merged 2026-11-12: a separate tiny group caused non-convergence in bootstrap resamples)

REF = {"race_cat": "White", "adm_type_cat": "Emergency", "icu_type": "Medical",
       "anchor_year_group": "2008 - 2010", "language_cat": "English"}

def prep(raw, exposure_levels):
    d = raw[raw["ins_cat"].isin(exposure_levels)].copy()
    d["icu_type"]    = d["first_careunit"].map(ICU_MAP).fillna("Mixed")
    d["female"]      = (d["gender"] == "F").astype(int)
    d["age_c"]       = (d["age_at_admit"] - 50) / 10            # per 10 years, centred at 50
    d["log_icu_los"] = np.log1p(d["icu_los_days_total"].astype(float))
    d["vent"]        = d["invasive_vent"].astype(int)
    d["pressor"]     = d["vasopressor"].astype(int)
    d["any_stay"]    = d["any_post_icu_stay"].astype(int)
    d["days_total"]  = np.where(d["any_stay"] == 1, d["post_icu_days"].astype(float), 0.0)  # two-part: <=4 h counted as 0
    d["post_icu_days"] = d["post_icu_days"].astype(float)
    d["sofa_day1"]   = d["sofa_day1"].astype(float)
    d["cci_no_age"]  = d["cci_no_age"].astype(float)
    d["facility"]    = d["dc_group"].isin(["snf", "rehab", "ltach"]).astype(int)
    for c in ["dc_ama", "dc_acute_transfer", "time_valid_strict"]:
        d[c] = d[c].astype(bool)
    d["dc_code"] = pd.Categorical(d["dc_group"], categories=DC_LEVELS).codes
    d["ins"] = pd.Categorical(d["ins_cat"], categories=exposure_levels)   # first level = reference
    for c, ref in REF.items():                                            # fixed level order, reference first
        levels = [ref] + sorted(x for x in d[c].dropna().unique() if x != ref)
        d[c] = pd.Categorical(d[c], categories=levels)
    n_missing = d[["sofa_day1", "cci_no_age", "dc_code"]].isna().sum()
    assert (d["dc_code"] >= 0).all(), "unexpected discharge group"
    print(f"n = {len(d):,} | missing:", n_missing.to_dict())
    return d.dropna(subset=["sofa_day1", "cci_no_age"]).reset_index(drop=True)

COV = ("age_c + I(age_c**2) + female + C(race_cat) + sofa_day1 + cci_no_age + C(adm_type_cat)"
       " + vent + pressor + log_icu_los + C(icu_type) + C(anchor_year_group)")
COV_LANG = COV + " + C(language_cat)"
EXPO = "C(ins)"

d18 = prep(raw18, ["Private", "Medicaid"])
d18["medicaid"] = (d18["ins_cat"] == "Medicaid").astype(int)
print(d18["ins"].value_counts())

In [ ]:
import warnings, numpy as np, pandas as pd, statsmodels.formula.api as smf
warnings.filterwarnings("ignore")   # the vent term is constant inside the ventilation cohort

rawB = client.query(f"SELECT * FROM `{PROJECT}.ins_icu.aim1_primary_18_64`").to_dataframe()
dB = prep(rawB, ["Private", "Medicaid"])
dB["medicaid"] = (dB["ins_cat"] == "Medicaid").astype(int)
adm = pd.to_datetime(dB["admittime"]).dt.normalize()
dis = pd.to_datetime(dB["dischtime"]).dt.normalize()
dB["los_tx"] = ((dis - adm).dt.days).clip(lower=1).astype(float)      # Texas LENGTH_OF_STAY rule
dB["y_snf"] = (dB["dc_group"] == "snf").astype(float) * 100
dB["y_rehab"] = (dB["dc_group"] == "rehab").astype(float) * 100
dB["y_facility"] = dB["dc_group"].isin(["snf", "rehab", "ltach"]).astype(float) * 100
dB["y_home"] = (dB["dc_group"] == "home").astype(float) * 100
dB["cci_cat"] = pd.cut(dB["cci_no_age"], [-1, 0, 1, 2, 4, 99], labels=["0", "1", "2", "3-4", "5+"]).astype(str)
dB["age_band"] = pd.cut(dB["age_at_admit"], [17, 24, 34, 44, 54, 64]).astype(str)   # Texas age groups
print(f"n = {len(dB):,}; Medicaid {dB['medicaid'].sum():,}")

COV_FULL = COV                                             # the paper's covariate set
COV_TX = "C(age_band) + female + C(race_cat) + C(adm_type_cat) + C(cci_cat) + vent + C(anchor_year_group)"

rows = []
for cohort, sub in [("All ICU survivors 18-64", dB), ("Invasive ventilation", dB[dB["vent"] == 1])]:
    for spec, cov in [("Paper covariates", COV_FULL), ("Texas-comparable covariates", COV_TX)]:
        for y in ["y_snf", "y_rehab", "y_facility", "y_home", "los_tx"]:
            r = smf.ols(f"{y} ~ medicaid + {cov}", data=sub).fit(cov_type="HC1")
            b, se = r.params["medicaid"], r.bse["medicaid"]
            rows.append(dict(cohort=cohort, covariates=spec, outcome=y, n=int(r.nobs),
                             private_mean=sub.loc[sub.medicaid == 0, y].mean(),
                             medicaid_mean=sub.loc[sub.medicaid == 1, y].mean(),
                             diff=b, lo=b - 1.96 * se, hi=b + 1.96 * se))
bridge = pd.DataFrame(rows)
print(bridge.round(2).to_string(index=False))
bridge.to_excel("mimic_bridge_results.xlsx", index=False)      # aggregate only; download and share

In [ ]:
from google.colab import files
files.download('mimic_bridge_results.xlsx')
